In [4]:
import os
import sys
import logging
import pandas as pd
import torch

In [5]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("whisper_finetune")

In [6]:
AUDIO_DIR = "/content/dataset/audio dataset"
TRANSCRIPT_CSV = "/content/dataset/transcripts.csv"
DRAFT_CSV = "/content/dataset/transcripts_draft.csv"
OUTPUT_DIR = "./whisper-finetuned-ar"
BASE_MODEL = "openai/whisper-small"
LANGUAGE = "arabic"

print("data loading is done")

data loading is done


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
log.info(f"Device: {device}")
if device == "cpu":
    log.warning("")

!pip install -q git+https://github.com/openai/whisper.git datasets transformers accelerate evaluate jiwer soundfile librosa

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.4 MB/s eta 0:00:00


In [8]:
from google.colab import files as colab_files
import zipfile
import os

uploaded = colab_files.upload()
zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print("Dataset extracted successfully!")

Saving audio dataset.zip to audio dataset.zip
Dataset extracted successfully!


In [9]:
import subprocess
import os

CONVERTED_DIR = "/content/dataset/audio_converted"
os.makedirs(CONVERTED_DIR, exist_ok=True)

audio_files = [f for f in os.listdir(AUDIO_DIR) if os.path.isfile(os.path.join(AUDIO_DIR, f))]

converted_files = []
failed_files = []

for i, f in enumerate(audio_files, 1):
    src_path = os.path.join(AUDIO_DIR, f)
    # اسم موحد بامتداد wav
    new_name = os.path.splitext(f)[0] + ".wav"
    dst_path = os.path.join(CONVERTED_DIR, new_name)

    try:
        result = subprocess.run(
            ["ffmpeg", "-y", "-i", src_path, "-ar", "16000", "-ac", "1", dst_path],
            capture_output=True, text=True, timeout=60
        )
        if result.returncode == 0 and os.path.exists(dst_path):
            converted_files.append(new_name)
            print(f"[{i}/{len(audio_files)}] ✅ {f} -> {new_name}")
        else:
            failed_files.append(f)
            print(f"[{i}/{len(audio_files)}] ❌ فشل: {f}")
            print(result.stderr[-300:])  # آخر جزء من رسالة الخطأ
    except Exception as e:
        failed_files.append(f)
        print(f"[{i}/{len(audio_files)}] ❌ استثناء: {f} -> {e}")

print(f"\nتم تحويل: {len(converted_files)} ملف")
print(f"فشل: {len(failed_files)} ملف")
if failed_files:
    print("الملفات اللي فشلت:", failed_files)

[1/81] ✅ WhatsApp Ptt 2026-09-07 at 2.01.50 AM.ogg -> WhatsApp Ptt 2026-09-07 at 2.01.50 AM.wav
[2/81] ✅ مصري-2026-08-26-12-04-روح-شمال.mp3 -> مصري-2026-08-26-12-04-روح-شمال.wav
[3/81] ✅ voice_06-09-2026_23-24-23 (4) -> voice_06-09-2026_23-24-23 (4).wav
[4/81] ✅ WhatsApp Ptt 2026-09-14 at 3.08.44 PM.ogg -> WhatsApp Ptt 2026-09-14 at 3.08.44 PM.wav
[5/81] ✅ voice_06-09-2026_23-15-47 -> voice_06-09-2026_23-15-47.wav
[6/81] ✅ WhatsApp Ptt 2026-09-07 at 2.02.58 AM.ogg -> WhatsApp Ptt 2026-09-07 at 2.02.58 AM.wav
[7/81] ✅ WhatsApp Ptt 2026-09-12 at 6.55.39 PM.ogg -> WhatsApp Ptt 2026-09-12 at 6.55.39 PM.wav
[8/81] ✅ صوت-راجل-مصري-2026-08-26-12-00-اطلع-لقدام.mp3 -> صوت-راجل-مصري-2026-08-26-12-00-اطلع-لقدام.wav
[9/81] ✅ WhatsApp Ptt 2026-09-14 at 3.10.53 PM.ogg -> WhatsApp Ptt 2026-09-14 at 3.10.53 PM.wav
[10/81] ✅ صوت-راجل-مصري-2026-08-26-11-59-شمال.mp3 -> صوت-راجل-مصري-2026-08-26-11-59-شمال.wav
[11/81] ✅ بنت-مصريه-2026-08-26-12-08-روح-يمين.mp3 -> بنت-مصريه-2026-08-26-12-08-روح-يمين.wav
[12/

In [10]:
AUDIO_DIR = CONVERTED_DIR

audio_files = sorted([
    f for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(".wav")
])
print("عدد الملفات بعد التوحيد:", len(audio_files))

عدد الملفات بعد التوحيد: 81


In [11]:
EXTRACT_DIR = "/content/dataset"
TRANSCRIPT_CSV = os.path.join(EXTRACT_DIR, "transcripts.csv")
DRAFT_CSV = os.path.join(EXTRACT_DIR, "transcripts_draft.csv")
OUTPUT_DIR = "./whisper-finetuned-ar"
BASE_MODEL = "openai/whisper-small"
LANGUAGE = "arabic"

In [12]:
for root, dirs, fs in os.walk('/content/dataset'):
    level = root.replace('/content/dataset', '').count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root) or root}/")
    for f in fs[:5]:
        print(f"{indent}  {f}")

dataset/
  audio_converted/
    مصرية-2026-08-26-12-12-شمال.wav
    صوت-راجل-مصري-2026-08-26-11-59-ارجع.wav
    voice_06-09-2026_23-24-15 (1).wav
    مصري-2026-08-26-12-04-روح-يمين.wav
    WhatsApp Ptt 2026-09-07 at 2.02.30 AM.wav
  audio dataset/
    WhatsApp Ptt 2026-09-07 at 2.01.50 AM.ogg
    مصري-2026-08-26-12-04-روح-شمال.mp3
    voice_06-09-2026_23-24-23 (4)
    WhatsApp Ptt 2026-09-14 at 3.08.44 PM.ogg
    voice_06-09-2026_23-15-47


In [13]:
if os.path.exists(DRAFT_CSV):
    os.remove(DRAFT_CSV)
    print("تم حذف الملف القديم الفاضي")

!apt-get -qq install -y ffmpeg
print("تم التأكد من ffmpeg")

تم التأكد من ffmpeg


In [14]:
import whisper
import traceback

test_model = whisper.load_model("medium", device=device)

test_file = os.path.join(AUDIO_DIR, audio_files[0])
print("بيتم اختبار الملف:", test_file)
print("الملف موجود؟", os.path.exists(test_file))
print("حجم الملف:", os.path.getsize(test_file), "بايت")

try:
    result = test_model.transcribe(test_file, language="ar")
    print("نجح! النص:", result["text"])
except Exception as e:
    print("فشل! الخطأ الكامل:")
    traceback.print_exc()

100%|██████████████████████████████████████| 1.42G/1.42G [00:12<00:00, 120MiB/s]


بيتم اختبار الملف: /content/dataset/audio_converted/WhatsApp Ptt 2026-09-07 at 2.01.50 AM.wav
الملف موجود؟ True
حجم الملف: 84350 بايت
نجح! النص: 


In [15]:
import traceback

# استخدم نفس الموديل اللي حمّلناه بالفعل (test_model) عشان نوفر وقت تحميل تاني
records = []
for i, f in enumerate(audio_files, 1):
    path = os.path.join(AUDIO_DIR, f)
    try:
        result = test_model.transcribe(path, language="ar")
        text = result["text"].strip()
        records.append({"file_name": f, "text": text})
        print(f"[{i}/{len(audio_files)}] ✅ {f} -> {text[:60]}")
    except Exception as e:
        print(f"[{i}/{len(audio_files)}] ❌ فشل: {f}")
        traceback.print_exc()

df = pd.DataFrame(records)
df.to_csv(DRAFT_CSV, index=False, encoding="utf-8-sig")
print("\nعدد الصفوف الناتجة:", len(df))
print(df)

[1/81] ✅ WhatsApp Ptt 2026-09-07 at 2.01.50 AM.wav -> 
[2/81] ✅ WhatsApp Ptt 2026-09-07 at 2.02.07 AM.wav -> إلى الأمام
[3/81] ✅ WhatsApp Ptt 2026-09-07 at 2.02.30 AM.wav -> وراء
[4/81] ✅ WhatsApp Ptt 2026-09-07 at 2.02.49 AM.wav -> يمين
[5/81] ✅ WhatsApp Ptt 2026-09-07 at 2.02.58 AM.wav -> يصور
[6/81] ✅ WhatsApp Ptt 2026-09-07 at 2.03.03 AM.wav -> هل تقصد؟
[7/81] ✅ WhatsApp Ptt 2026-09-07 at 2.03.08 AM.wav -> شمال
[8/81] ✅ WhatsApp Ptt 2026-09-12 at 6.55.32 PM.wav -> او ده
[9/81] ✅ WhatsApp Ptt 2026-09-12 at 6.55.34 PM.wav -> ورا
[10/81] ✅ WhatsApp Ptt 2026-09-12 at 6.55.39 PM.wav -> يمين
[11/81] ✅ WhatsApp Ptt 2026-09-12 at 6.55.43 PM.wav -> شمان
[12/81] ✅ WhatsApp Ptt 2026-09-12 at 6.56.07 PM.wav -> اتحرك قدام
[13/81] ✅ WhatsApp Ptt 2026-09-12 at 6.56.11 PM.wav -> ارغاب وراء
[14/81] ✅ WhatsApp Ptt 2026-09-12 at 6.56.15 PM.wav -> لف يمين
[15/81] ✅ WhatsApp Ptt 2026-09-12 at 6.56.19 PM.wav -> لف شمال
[16/81] ✅ WhatsApp Ptt 2026-09-14 at 3.08.32 PM.wav -> وداعا
[17/81] ✅ WhatsApp Ptt 2

In [16]:
if not df.empty:
    missing_files = [f for f in df["file_name"] if not os.path.exists(os.path.join(AUDIO_DIR, f))]
    if missing_files:
        log.warning(f"{len(missing_files)} listed files are missing and will be removed.")
        df = df[~df["file_name"].isin(missing_files)]

    empty_texts = df["text"].isna().sum() + (df["text"].astype(str).str.strip() == "").sum()
    if empty_texts > 0:
        log.warning(f"There are {empty_texts} empty texts; they will be removed.")
        df = df[df["text"].notna() & (df["text"].astype(str).str.strip() != "")]

    log.info(f"Number of valid training samples: {len(df)}")
    if len(df) < 10:
        log.warning("The number of samples is very small (<10) — the results will be weak.")
else:
    log.warning("DataFrame is empty, skipping data cleaning and validation.")

In [17]:
from datasets import Dataset, Audio

df["audio"] = df["file_name"].apply(lambda x: os.path.join(AUDIO_DIR, x))
dataset = Dataset.from_pandas(df[["audio", "text"]].reset_index(drop=True))
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

test_size = 0.1 if len(df) >= 20 else max(1, int(len(df) * 0.1))
dataset = dataset.train_test_split(test_size=test_size, seed=42)
log.info(f"Train: {len(dataset['train'])} | Test: {len(dataset['test'])}")

In [18]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language=LANGUAGE, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.generation_config.language = LANGUAGE
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.to(device)

print("تم تحميل الموديل:", BASE_MODEL)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

تم تحميل الموديل: openai/whisper-small


In [19]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

log.info("جاري تجهيز الـ features...")
dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, num_proc=1)
print("تم تجهيز الـ features بنجاح")

Map (num_proc=1):   0%|          | 0/72 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/8 [00:00<?, ? examples/s]

تم تجهيز الـ features بنجاح


In [20]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("تم إعداد الـ Data Collator")

تم إعداد الـ Data Collator


In [21]:
import evaluate
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("تم إعداد مقياس WER")

تم إعداد مقياس WER


In [22]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

n_samples = len(dataset["train"])
batch_size = 8 if n_samples > 100 else max(1, min(4, n_samples))
max_steps = min(1000, max(100, n_samples * 20))

log.info(f"Batch size: {batch_size} | Max steps: {max_steps}")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=max(1, 16 // batch_size),
    learning_rate=1e-5,
    warmup_steps=min(100, max_steps // 10),
    max_steps=max_steps,
    gradient_checkpointing=True,
    fp16=(device == "cuda"),
    eval_strategy="steps",
    per_device_eval_batch_size=max(1, batch_size // 2),
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=max(50, max_steps // 5),
    eval_steps=max(50, max_steps // 5),
    logging_steps=25,
    report_to=["none"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("تم إعداد الـ Trainer")

تم إعداد الـ Trainer


In [ ]:
log.info("بدء التدريب...")
try:
    trainer.train()
except Exception as e:
    log.error(f"حصل خطأ أثناء التدريب: {e}")
    raise

Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Wer
200,0.017389,0.798260,63.636364
400,0.009067,0.868648,54.545455
600,0.008575,0.883114,54.545455


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
final_dir = f"{OUTPUT_DIR}-final"
model.save_pretrained(final_dir)
processor.save_pretrained(final_dir)
log.info(f"تم حفظ الموديل في: {final_dir}")

In [ ]:
import librosa

def transcribe_audio(audio_path: str) -> str:
    speech, sr = librosa.load(audio_path, sr=16000)
    input_features = processor.feature_extractor(speech, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    with torch.no_grad():
        predicted_ids = model.generate(input_features)
    return processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]

test_file = os.path.join(AUDIO_DIR, audio_files[0])
log.info(f"اختبار على: {audio_files[0]}")
print("النص المتوقع:", transcribe_audio(test_file))

In [ ]:
final_dir = "/content/drive/MyDrive/whisper-finetuned-ar-final"
model.save_pretrained(final_dir)
processor.save_pretrained(final_dir)
print("تم الحفظ في Google Drive:", final_dir)

In [ ]:
!git clone https://github.com/ggerganov/whisper.cpp.git /content/whisper.cpp
%cd /content/whisper.cpp
!make